In [0]:
df=spark.read.table('default.crm_cust_info')
display(df)

In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col
for fields in df.schema:
  if isinstance(fields.dataType, StringType):
            df = df.withColumn(fields.name, trim(col(fields.name)))

In [0]:
display(df)

In [0]:
from pyspark.sql import functions as F
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)
display(df)

In [0]:
df=df.filter(df.cst_id.isNotNull())
display(df)

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
display(df)

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.default.crm_customers")